# 2.1 理化指标与 quality 的探索分析 · 复核

这一步当时用 `src/analyze.py` 执行，产物是 `outputs/eda.json`（全部 11 个指标的分布、相关系数、分组中位数、同值行敏感性）。
这个 notebook 从原始 CSV 重算其中最关键的几项，和当时的产物对账，不改任何历史文件。


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
WINES = {"red": "红酒", "white": "白酒"}
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
STEP = ROOT / "steps/02_EDA/2.1_理化指标与quality的探索分析"
eda = json.loads((STEP / "outputs/eda.json").read_text(encoding="utf-8"))
run = dsflow.start_run("2.1", project=ROOT, hypothesis="从原始 CSV 重算各指标与 quality 的 Spearman、分组中位数和同值行敏感性，与 2.1 当时的产物一致")
raw = {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";")
print("红酒 %d 行、白酒 %d 行，全部原始行，不删同值行" % (len(raw["red"]), len(raw["white"])))


红酒 1599 行、白酒 4898 行，全部原始行，不删同值行


In [2]:
rows = []
for w, name in WINES.items():
    df = raw[w]
    for feat in FEATURES:
        sp = float(df[feat].corr(df["quality"], method="spearman"))   # 并列值取平均秩
        ref = eda["files"][w]["indicators"][feat]["spearman_quality"]
        assert abs(sp - ref) < 1e-9, (w, feat, sp, ref)
        rows.append({"酒类": name, "指标": feat, "与 quality 的 Spearman": round(sp, 4)})
table = pd.DataFrame(rows).pivot(index="指标", columns="酒类", values="与 quality 的 Spearman").loc[FEATURES]
print(table.to_string())
print("22 个 Spearman 系数与 2.1 当时的产物一致（容差 1e-9）。")


酒类                        白酒      红酒
指标                                  
fixed acidity        -0.0845  0.1141
volatile acidity     -0.1966 -0.3806
citric acid           0.0183  0.2135
residual sugar       -0.0821  0.0320
chlorides            -0.3145 -0.1899
free sulfur dioxide   0.0237 -0.0569
total sulfur dioxide -0.1967 -0.1967
density              -0.3484 -0.1771
pH                    0.1094 -0.0437
sulphates             0.0333  0.3771
alcohol               0.4404  0.4785
22 个 Spearman 系数与 2.1 当时的产物一致（容差 1e-9）。


In [3]:
for w, name in WINES.items():
    df = raw[w]
    med = df.groupby("quality")["alcohol"].agg(行数="size", alcohol中位数="median")
    for q, r in med.iterrows():
        ref = eda["files"][w]["indicators"]["alcohol"]["by_quality"][str(int(q))]
        assert int(r["行数"]) == ref["rows"] and abs(r["alcohol中位数"] - ref["median"]) < 1e-9
    print(f"{name}：各 quality 分组的 alcohol 中位数")
    print(med.to_string())


红酒：各 quality 分组的 alcohol 中位数
          行数  alcohol中位数
quality                 
3         10       9.925
4         53      10.000
5        681       9.700
6        638      10.500
7        199      11.500
8         18      12.150
白酒：各 quality 分组的 alcohol 中位数
           行数  alcohol中位数
quality                  
3          20       10.45
4         163       10.10
5        1457        9.50
6        2198       10.50
7         880       11.40
8         175       12.00
9           5       12.50


In [4]:
rows = []
for w, name in WINES.items():
    df = raw[w]
    tx = pd.read_csv(ROOT / f"data/winequality-{w}.csv", sep=";", dtype=str)
    keep = ~tx.duplicated()                          # 同值行每组只留首次出现
    uniq = df[keep]
    for feat in ("alcohol", "chlorides", "volatile acidity"):
        a = float(df[feat].corr(df["quality"], method="spearman"))
        u = float(uniq[feat].corr(uniq["quality"], method="spearman"))
        ref = eda["files"][w]["sensitivity"]["by_indicator"][feat]
        assert abs((u - a) - ref["difference_unique_minus_all"]) < 1e-9
        rows.append({"酒类": name, "指标": feat, "全部行": len(df), "只留首次的行": len(uniq), "Spearman 全部行": round(a, 4), "Spearman 只留首次": round(u, 4), "差": round(u - a, 4)})
sens = pd.DataFrame(rows)
print(sens.to_string(index=False))
run.log_metrics({"红酒_alcohol_Spearman": float(raw["red"]["alcohol"].corr(raw["red"]["quality"], method="spearman")),
                 "白酒_alcohol_Spearman": float(raw["white"]["alcohol"].corr(raw["white"]["quality"], method="spearman"))})
run.set_conclusion("22 个 Spearman、alcohol 分组中位数、同值行敏感性差值与 2.1 当时的 eda.json 一致", validity="有效")
run.end()


酒类               指标  全部行  只留首次的行  Spearman 全部行  Spearman 只留首次       差
红酒          alcohol 1599    1359        0.4785         0.4880  0.0094
红酒        chlorides 1599    1359       -0.1899        -0.2044 -0.0144
红酒 volatile acidity 1599    1359       -0.3806        -0.3874 -0.0068
白酒          alcohol 4898    3961        0.4404         0.4757  0.0353
白酒        chlorides 4898    3961       -0.3145        -0.3331 -0.0186
白酒 volatile acidity 4898    3961       -0.1966        -0.1854  0.0112
